# Исследование данных Titanic
Практические работы 4–6: подготовка данных, корреляционный анализ, построение и оценка моделей классификации (`Survived`).

## Загрузка файла
Запусти ячейку и выбери `titanic.csv` (если файл уже в Colab — ячейку можно пропустить).

In [ ]:
from google.colab import files
import os
if not os.path.exists("titanic.csv"):
    up = files.upload()
    name = list(up.keys())[0]
    os.rename(name, "titanic.csv")

## Часть 4. Подготовка данных

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("titanic.csv")
print(df.head())

features = ["Pclass", "Age", "Sex", "Fare", "SibSp", "Parch"]
X = df[features].copy()
y = df["Survived"]

X["Age"] = X["Age"].fillna(X["Age"].median())
X["Sex"] = X["Sex"].map({"male": 0, "female": 1})

print(X.isnull().sum())
print(X.dtypes)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Размер X_train:", X_train.shape)
print("Размер X_test:", X_test.shape)

**Задание 1.** Пропусков в выбранных признаках нет (сумма пропусков = 0). Размеры выборок: `X_train` — (712, 6), `X_test` — (179, 6), то есть 80% / 20% от 891 пассажира.

**Вопрос: зачем нужна отдельная тестовая выборка и почему `fit_transform()` только для обучающей?**
Тестовая выборка имитирует новые, ранее не виденные данные — на ней честно проверяется, как модель обобщает. Если оценивать на данных обучения, результат будет завышен. `fit_transform()` вычисляет среднее и стандартное отклонение, поэтому его применяют только к train: иначе информация о тесте «просочится» в обучение (утечка данных). К тесту применяют `transform()` с параметрами, найденными по train.

## Часть 5. Корреляционный анализ

In [ ]:
data_corr = X.copy()
data_corr["Survived"] = y
corr = data_corr.corr()
print(corr["Survived"].sort_values(ascending=False))

plt.figure(figsize=(7, 5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Корреляционная матрица")
plt.show()

**Выводы по корреляции с `Survived`:**
- Наиболее сильная положительная: **Sex** (0.54) — женщины выживали намного чаще (1 = female).
- Наиболее сильная отрицательная: **Pclass** (−0.34) — чем ниже класс (больше номер), тем ниже шанс выжить.
- Fare (0.26) положительно связан с выживанием — дорогие билеты, как правило, 1 класс.
- Age, SibSp, Parch — слабая связь.

Корреляция показывает только линейную связь и не доказывает причинность.

## Часть 6. Логистическая регрессия

In [ ]:
from sklearn.linear_model import LogisticRegression

model_lr = LogisticRegression(max_iter=1000)
model_lr.fit(X_train_scaled, y_train)

**Почему обучаем на `X_train_scaled` и `y_train`?** Модель должна учиться только на обучающих данных. Тест нужен для независимой проверки; если обучать на нём, оценка станет нечестной.

In [ ]:
y_pred_lr = model_lr.predict(X_test_scaled)

print("Первые предсказания:", y_pred_lr[:10])
print("Правильные ответы:", y_test.iloc[:10].to_list())

**Задание 3.** Предсказания: `[0 0 0 0 1 0 1 0 0 0]`, правильные: `[0, 0, 1, 0, 1, 1, 1, 0, 0, 0]`.
Совпали, например, 1-й, 2-й, 4-й, 5-й, 7-й ответы. Ошибки: 3-й пассажир (предсказано 0, на деле выжил) и 6-й (предсказано 0, на деле выжил).
**0** — пассажир не выжил, **1** — выжил.

In [ ]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)

print("Accuracy:", accuracy_score(y_test, y_pred_lr))
print("Precision:", precision_score(y_test, y_pred_lr))
print("Recall:", recall_score(y_test, y_pred_lr))
print("F1:", f1_score(y_test, y_pred_lr))
print("Матрица ошибок:\n", confusion_matrix(y_test, y_pred_lr))
print(classification_report(y_test, y_pred_lr))

**Задание 4. Метрики логистической регрессии:** Accuracy = 0.804, Precision = 0.766, Recall = 0.710, F1 = 0.737.
Матрица `[[TN, FP], [FN, TP]]` = `[[95, 15], [20, 49]]`.

- **Доля правильных предсказаний** — accuracy ≈ 80,4% (144 из 179).
- **Насколько модель находит реально выживших** — recall (полнота), 71%: нашла 49 из 69 выживших.
- **Ложноположительное (FP)** — модель предсказала «выжил», а пассажир погиб (15 случаев). **Ложноотрицательное (FN)** — предсказала «не выжил», а пассажир выжил (20 случаев).

## Дерево решений

In [ ]:
from sklearn.tree import DecisionTreeClassifier

model_tree = DecisionTreeClassifier(max_depth=4, random_state=42)
model_tree.fit(X_train, y_train)

y_pred_tree = model_tree.predict(X_test)

print("Accuracy дерева:", accuracy_score(y_test, y_pred_tree))
print("Precision:", precision_score(y_test, y_pred_tree))
print("Recall:", recall_score(y_test, y_pred_tree))
print("F1:", f1_score(y_test, y_pred_tree))
print("Матрица ошибок:\n", confusion_matrix(y_test, y_pred_tree))

res = pd.DataFrame({
    "Логистическая регрессия": [accuracy_score(y_test, y_pred_lr), precision_score(y_test, y_pred_lr),
                                recall_score(y_test, y_pred_lr), f1_score(y_test, y_pred_lr)],
    "Дерево решений": [accuracy_score(y_test, y_pred_tree), precision_score(y_test, y_pred_tree),
                       recall_score(y_test, y_pred_tree), f1_score(y_test, y_pred_tree)],
}, index=["Accuracy", "Precision", "Recall", "F1"]).round(3)
print(res)

imp = pd.Series(model_tree.feature_importances_, index=features).sort_values()
imp.plot(kind="barh", title="Важность признаков (дерево)")
plt.show()

**Задание 5. Сравнение моделей** (одна и та же тестовая выборка):

| Метрика | Логистическая регрессия | Дерево решений |
|---|---|---|
| Accuracy | 0.804 | 0.782 |
| Precision | 0.766 | 0.875 |
| Recall | 0.710 | 0.507 |
| F1 | 0.737 | 0.642 |

Матрица дерева: `[[105, 5], [34, 35]]`.

Логистическая регрессия лучше по accuracy, recall и F1. Дерево реже ошибочно говорит «выжил» (precision выше), но пропускает много реально выживших (34 FN, recall всего 51%).
Важность признаков в дереве: Sex (0.59), Pclass (0.20), Fare (0.10), Age (0.09) — это согласуется с корреляционным анализом.

## Итоговые выводы
- Главные факторы выживания: **пол** (женщины выживали чаще) и **класс билета** (1 класс — выше шансы). Возраст, число родственников — слабее.
- Лучшая модель по сбалансированным метрикам — **логистическая регрессия**: accuracy ≈ 80%, F1 ≈ 0.74.
- Модель неплохо предсказывает выживаемость, но не идеально: она находит около 71% выживших. Можно улучшать, добавив признаки (например, `Embarked`, титул из `Name`), подбирая гиперпараметры и используя кросс-валидацию.